# ДЗ07 — Domain shift RGB→IR (VisDrone → HIT-UAV)

**Автор:** Serhii Polishchuk  

| Крок | Зміст |
|------|--------|
| 1 | Два мапінги (eval / train) |
| 2 | Zero-shot на HIT val |
| 3 | Інтервенції A, B, A+B |
| 4 | Зведена таблиця + small objects |
| 5 | Висновок |

Порядок: **zero → A → B → A+B**. Один evaluator (imgsz/conf/iou).

In [1]:
!pip install -q ultralytics opencv-python-headless pyyaml

import os, json, random, shutil, hashlib, zipfile, urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import yaml
import torch
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMGSZ = 640
CONF = 0.25
IOU = 0.7
EPOCHS_A = 15   # grayscale VisDrone subset
EPOCHS_B = 20   # HIT subset
EPOCHS_AB = 15  # HIT from weights A
SUBSET_N = 150  # 100–200 кадрів

OUT = Path("./dz07_out")
OUT.mkdir(exist_ok=True)
(OUT / "weights").mkdir(exist_ok=True)
(OUT / "lists").mkdir(exist_ok=True)
(OUT / "viz").mkdir(exist_ok=True)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
CUDA: True
Tesla T4


In [2]:
!mkdir -p /content/datasets
!wget -O /content/HIT-UAV.zip \
  "https://zenodo.org/api/records/7633134/files/suojiashun/HIT-UAV-Infrared-Thermal-Dataset-v1.2.1.zip/content"

!ls -lh /content/HIT-UAV.zip
!unzip -q /content/HIT-UAV.zip -d /content/datasets/HIT-UAV
!ls /content/datasets/HIT-UAV | head

--2026-11-10 08:55:49--  https://zenodo.org/api/records/7633134/files/suojiashun/HIT-UAV-Infrared-Thermal-Dataset-v1.2.1.zip/content
Resolving zenodo.org (zenodo.org)... 188.185.42.113, 188.185.40.132, 188.185.45.63, ...
Connecting to zenodo.org (zenodo.org)|188.185.42.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 814751578 (777M) [application/octet-stream]
Saving to: ‘/content/HIT-UAV.zip’

/content/HIT-UAV.zi 100%[===================>] 777.01M  2.18MB/s    in 6m 57s  

2026-11-10 09:02:46 (1.86 MB/s) - ‘/content/HIT-UAV.zip’ saved [814751578/814751578]

-rw-r--r-- 1 root root 778M Nov 10 09:02 /content/HIT-UAV.zip
suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c


In [3]:
!ls -lh /content/HIT-UAV.zip
!ls /content/datasets
!ls /content/datasets/HIT-UAV 2>/dev/null | head
!find /content -maxdepth 3 -type d -iname "*hit*" 2>/dev/null

-rw-r--r-- 1 root root 778M Nov 10 09:02 /content/HIT-UAV.zip
HIT-UAV
suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c
/content/datasets/HIT-UAV
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c


In [4]:
!mkdir -p /content/datasets/HIT-UAV
!unzip -q -o /content/HIT-UAV.zip -d /content/datasets/HIT-UAV
!ls /content/datasets/HIT-UAV | head

suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c


In [5]:
!ls "/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c"
!find "/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c" -maxdepth 2 -type d

0_readme_images  LICENSE      normal_xml  rotate_json  tools
CITATION.cff	 normal_json  README.md   rotate_xml
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_xml
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_xml/ImageSets
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_xml/JPEGImages
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_xml/Annotations
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/rotate_xml
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/rotate_xml/ImageSets
/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/rotate_xml/JPEGImages
/content/datasets/HIT-UAV/suojiashun-HIT

In [6]:
HIT_ROOT = Path("/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c")

!ls "{HIT_ROOT}/normal_xml/JPEGImages" | head
!ls "{HIT_ROOT}/normal_xml/JPEGImages" | wc -l
!ls "{HIT_ROOT}/normal_json"
!ls "{HIT_ROOT}/normal_xml/ImageSets" 2>/dev/null

0_100_30_0_03280.jpg
0_100_30_0_03281.jpg
0_100_30_0_03284.jpg
0_100_30_0_03288.jpg
0_100_30_0_03289.jpg
0_100_30_0_03291.jpg
0_100_30_0_03292.jpg
0_100_30_0_03294.jpg
0_100_30_0_03298.jpg
0_100_30_0_03301.jpg
2898
annotations  test  train  val
Main


## 1. Мапінги (фіксуємо до будь-якого запуску)

### Оцінювання: VisDrone pred → 3 класи HIT-UAV
| HIT-UAV | VisDrone class ids |
|---------|-------------------|
| Person | pedestrian(0), people(1) |
| Car | car(3) |
| Bicycle | bicycle(2) |

### Навчання: HIT category → YOLO class_id у мітках
| HIT cat | YOLO id у labels | VisDrone-аналог |
|---------|------------------|-----------------|
| Person(0) | **0** | pedestrian |
| Car(1) | **3** | car |
| Bicycle(2) | **2** | bicycle |
| OtherVehicle/DontCare | не пишемо | — |

У yaml names — як у ДЗ1 (повний COCO/VisDrone список або той самий), щоб не змінювалась голова.

In [7]:
# --- EVAL mapping: VisDrone cls → HIT report name ---
EVAL_VD_TO_HIT = {
    0: "Person",   # pedestrian
    1: "Person",   # people
    2: "Bicycle",  # bicycle
    3: "Car",      # car
    # van, truck, bus, motor, ... → ignore for 3-class metrics
}
HIT_NAMES_3 = ["Person", "Car", "Bicycle"]
HIT_NAME_TO_IDX = {"Person": 0, "Car": 1, "Bicycle": 2}

# --- TRAIN mapping: HIT category_id → YOLO class_id in label files ---
# HIT JSON often: 0 Person, 1 Car, 2 Bicycle, 3 OtherVehicle, 4 DontCare
TRAIN_HIT_TO_YOLO = {
    0: 0,  # Person → 0 (pedestrian slot)
    1: 3,  # Car → 3
    2: 2,  # Bicycle → 2
    # 3, 4 skipped
}

# Also string names if annotations use names
TRAIN_HIT_NAME_TO_YOLO = {
    "person": 0, "Person": 0,
    "car": 3, "Car": 3,
    "bicycle": 2, "Bicycle": 2,
}

print("EVAL map:", EVAL_VD_TO_HIT)
print("TRAIN map:", TRAIN_HIT_TO_YOLO)

EVAL map: {0: 'Person', 1: 'Person', 2: 'Bicycle', 3: 'Car'}
TRAIN map: {0: 0, 1: 3, 2: 2}


## 2. Дані: HIT-UAV + VisDrone

**HIT-UAV:** поклади розпакований датасет у `./datasets/HIT-UAV/`  
(з GitHub/Zenodo/Kaggle: images + annotations JSON або вже YOLO).  

Очікувані варіанти структури (ноутбук підхопить):
- `images/train`, `images/val` + `labels/train`, `labels/val` (YOLO)
- або COCO/JSON `annotations/` + `images/`

**VisDrone train** — для інтервенції A (grayscale subset).

In [8]:
HIT_ROOT = Path("./datasets/HIT-UAV")
VD_ROOT = Path("./datasets/VisDrone")

# --- VisDrone DET val/train (як у попередніх ДЗ) ---
def ensure_visdrone_val():
    target = VD_ROOT / "VisDrone2019-DET-val"
    if target.exists() and any(target.rglob("*.jpg")):
        return target
    VD_ROOT.mkdir(parents=True, exist_ok=True)
    url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
    zpath = VD_ROOT / "val.zip"
    print("Downloading VisDrone val...")
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(VD_ROOT)
    zpath.unlink(missing_ok=True)
    return target

vd_val = ensure_visdrone_val()
print("VisDrone val:", vd_val)

# HIT check
print("HIT-UAV root exists:", HIT_ROOT.exists())
if HIT_ROOT.exists():
    for p in sorted(HIT_ROOT.rglob("*"))[:20]:
        if p.is_file():
            print(" ", p.relative_to(HIT_ROOT))
else:
    print("""
>>> ПОКЛАДИ HIT-UAV у ./datasets/HIT-UAV/
Джерела:
  - https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset
  - Zenodo: https://zenodo.org/records/7633134
  - Kaggle: HIT-UAV YOLO format
Після розпаковки перезапусти цю клітинку.
""")

VisDrone val: datasets/VisDrone/VisDrone2019-DET-val
HIT-UAV root exists: True
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/1.jpg
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/2.jpg
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/3.jpg
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/4.jpg
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/5.jpg
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/6.jpg
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/oriented_box.png
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/0_readme_images/standard_box.png
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/CITATION.cff
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/LICENSE
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/README.md
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_json/.DS_Store
  suojiashun-HIT-UAV-I

In [9]:
def find_hit_split(root: Path):
    """Return dict train/val with image dirs and optional label dirs or json."""
    info = {}
    # YOLO layout
    for split in ["train", "val", "test"]:
        for cand in [root / "images" / split, root / split / "images", root / split]:
            if cand.exists() and any(cand.glob("*.jpg")) or any(cand.glob("*.png")):
                lab = None
                for lc in [root / "labels" / split, root / split / "labels", cand.parent / "labels"]:
                    if lc.exists():
                        lab = lc
                        break
                info[split] = {"images": cand, "labels": lab}
                break
    # flat images
    if not info:
        imgs = list(root.rglob("*.jpg")) + list(root.rglob("*.png"))
        if imgs:
            info["all"] = {"images": root, "labels": None, "files": imgs}
    return info

hit_info = find_hit_split(HIT_ROOT) if HIT_ROOT.exists() else {}
print("HIT splits found:", list(hit_info.keys()))
for k, v in hit_info.items():
    n = len(list(v["images"].glob("*.jpg"))) + len(list(v["images"].glob("*.png")))
    print(f"  {k}: {n} images, labels={v.get('labels')}")

HIT splits found: ['all']
  all: 0 images, labels=None


In [10]:
def convert_hit_json_to_yolo(json_path: Path, images_dir: Path, out_labels: Path):
    """COCO-like HIT JSON → YOLO labels with TRAIN remap (0/3/2). Skip Other/DontCare."""
    out_labels.mkdir(parents=True, exist_ok=True)
    data = json.loads(json_path.read_text())
    # build image id → file + size
    id2img = {}
    for im in data.get("images", []):
        id2img[im["id"]] = im
    # category id map from annotation categories
    cat_map = {}
    for c in data.get("categories", []):
        name = c.get("name", "")
        if name.lower() in ("person",):
            cat_map[c["id"]] = 0
        elif name.lower() in ("car",):
            cat_map[c["id"]] = 3
        elif name.lower() in ("bicycle",):
            cat_map[c["id"]] = 2
        # else skip

    by_img = defaultdict(list)
    for ann in data.get("annotations", []):
        cid = ann.get("category_id")
        yolo_cls = cat_map.get(cid)
        if yolo_cls is None:
            # try TRAIN_HIT_TO_YOLO on raw id
            yolo_cls = TRAIN_HIT_TO_YOLO.get(cid)
        if yolo_cls is None:
            continue
        im = id2img.get(ann["image_id"])
        if not im:
            continue
        W, H = im.get("width"), im.get("height")
        if not W or not H:
            continue
        # bbox xywh absolute
        x, y, w, h = ann["bbox"]
        xc, yc = (x + w / 2) / W, (y + h / 2) / H
        bw, bh = w / W, h / H
        by_img[im["file_name"]].append(f"{yolo_cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

    n = 0
    for fname, lines in by_img.items():
        stem = Path(fname).stem
        (out_labels / f"{stem}.txt").write_text("".join(lines))
        n += 1
    print(f"Converted {n} label files → {out_labels}")
    return out_labels

# If labels missing but JSON present — convert
if HIT_ROOT.exists():
    jsons = list(HIT_ROOT.rglob("*.json"))
    print("JSON files:", len(jsons))
    for jp in jsons[:5]:
        print(" ", jp.relative_to(HIT_ROOT))

JSON files: 7
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_json/annotations/train.json
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_json/annotations/val.json
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_json/annotations/test.json
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/rotate_json/annotations/train.json
  suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/rotate_json/annotations/val.json


In [11]:
# Побудова YOLO dataset layout для HIT val/train з remap
HIT_YOLO = Path("./datasets/HIT-UAV-yolo")

def build_hit_yolo_from_existing():
    """Copy/link images and ensure labels use TRAIN remap."""
    if not hit_info:
        print("No HIT data found")
        return None
    for split in ["train", "val", "test"]:
        if split not in hit_info:
            continue
        src_img = hit_info[split]["images"]
        src_lab = hit_info[split].get("labels")
        dst_img = HIT_YOLO / "images" / split
        dst_lab = HIT_YOLO / "labels" / split
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lab.mkdir(parents=True, exist_ok=True)

        imgs = list(src_img.glob("*.jpg")) + list(src_img.glob("*.png"))
        for ip in tqdm(imgs, desc=f"HIT {split}"):
            dst = dst_img / ip.name
            if not dst.exists():
                try:
                    os.symlink(ip.resolve(), dst)
                except Exception:
                    shutil.copy(ip, dst)
            # labels
            lp = None
            if src_lab:
                for ext in [".txt"]:
                    cand = src_lab / (ip.stem + ext)
                    if cand.exists():
                        lp = cand
                        break
            out_lp = dst_lab / (ip.stem + ".txt")
            if lp and not out_lp.exists():
                # remap class ids if labels are HIT-native 0,1,2
                lines_out = []
                for line in lp.read_text().strip().splitlines():
                    p = line.split()
                    if len(p) < 5:
                        continue
                    cid = int(float(p[0]))
                    # if already YOLO 0/2/3 keep; if HIT 0/1/2 remap
                    if cid in (0, 1, 2) and cid != 3:
                        # ambiguous: assume HIT ids if max class in file <= 4
                        yid = TRAIN_HIT_TO_YOLO.get(cid, None)
                        if yid is None:
                            continue
                        cid = yid
                    if cid not in (0, 2, 3):
                        continue
                    lines_out.append(f"{cid} " + " ".join(p[1:]) + "\n")
                out_lp.write_text("".join(lines_out))
            elif not out_lp.exists():
                out_lp.write_text("")  # empty allowed
    return HIT_YOLO

if HIT_ROOT.exists() and hit_info:
    build_hit_yolo_from_existing()
    print("HIT YOLO layout:", HIT_YOLO)
else:
    print("Skip HIT YOLO build — add dataset first")

HIT YOLO layout: datasets/HIT-UAV-yolo


In [12]:
# YAML з names як у VisDrone/COCO (не змінюємо голову)
# Для val HIT використовуємо images/val + labels з remap

def write_data_yaml(path, train_rel, val_rel, root):
    # names: full 80 COCO to keep head compatible with yolov8n pretrained / DZ1
    tmp = YOLO("yolov8n.pt")
    names_block = "\n".join([f"  {i}: {n}" for i, n in tmp.names.items()])
    text = f"""
path: {Path(root).resolve()}
train: {train_rel}
val: {val_rel}

names:
{names_block}
""".strip() + "\n"
    Path(path).write_text(text)
    print("Wrote", path)
    return path

HIT_YAML = OUT / "hit_uav.yaml"
if (HIT_YOLO / "images" / "val").exists():
    write_data_yaml(HIT_YAML, "images/train", "images/val", HIT_YOLO)
elif (HIT_YOLO / "images" / "test").exists():
    write_data_yaml(HIT_YAML, "images/train", "images/test", HIT_YOLO)
else:
    print("HIT yaml pending — dataset not ready")

HIT yaml pending — dataset not ready


## 3. Zero-shot (RGB→IR)

Ваги: checkpoint ДЗ1 (`best.pt`) або `yolov8n.pt` якщо немає.

Метрики рахуємо через `val`, потім **фільтруємо класи** до Person/Car/Bicycle за EVAL map.

In [14]:
WEIGHTS_DZ1 = "yolov8n.pt"
for c in [Path("/content/best.pt"), Path("./best.pt"),
          Path("../DZ03_Vision_Serhii Polishchuk/best.pt"),
          Path("./dz07_out/weights/best_dz1.pt")]:
    if c.exists():
        WEIGHTS_DZ1 = str(c)
        break
print("Zero-shot weights:", WEIGHTS_DZ1)

if not HIT_YAML.exists():
    print("STOP: спочатку підготуй HIT dataset і hit_uav.yaml")
else:
    model0 = YOLO(WEIGHTS_DZ1)
    res0 = model0.val(data=str(HIT_YAML), imgsz=IMGSZ, conf=CONF, iou=IOU,
                      split="val", plots=False, verbose=True)
    # overall box metrics (includes all classes model predicts)
    zero_metrics = {
        "run": "zero",
        "mAP50": float(res0.box.map50),
        "mAP50-95": float(res0.box.map),
        "precision": float(res0.box.mp),
        "recall": float(res0.box.mr),
    }
    # per-class if available
    names = model0.names
    if hasattr(res0.box, "ap_class_index") and res0.box.ap_class_index is not None:
        for i, ci in enumerate(res0.box.ap_class_index):
            cname = names.get(int(ci), str(ci))
            zero_metrics[f"AP50_{cname}"] = float(res0.box.ap50[i]) if hasattr(res0.box, "ap50") else None
    print("ZERO:", zero_metrics)
    pd.DataFrame([zero_metrics]).to_csv(OUT / "metrics_zero.csv", index=False)

Zero-shot weights: yolov8n.pt
STOP: спочатку підготуй HIT dataset і hit_uav.yaml


In [15]:
# 2–3 кадри viz: GT + pred
def viz_predictions(model, img_paths, out_dir, tag, n=3):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = list(img_paths)[:n]
    for ip in paths:
        r = model.predict(str(ip), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
        plotted = r.plot()  # BGR numpy
        cv2.imwrite(str(out_dir / f"{tag}_{Path(ip).stem}.jpg"), plotted)
        print("Saved", out_dir / f"{tag}_{Path(ip).stem}.jpg")

if HIT_YAML.exists() and (HIT_YOLO / "images" / "val").exists():
    val_imgs = sorted((HIT_YOLO / "images" / "val").glob("*.jpg"))[:50]
    if not val_imgs:
        val_imgs = sorted((HIT_YOLO / "images" / "val").glob("*.png"))[:50]
    viz_predictions(model0, val_imgs, OUT / "viz", "zero", n=3)

In [16]:
import xml.etree.ElementTree as ET
from pathlib import Path
import shutil, os

HIT_ROOT = Path("/content/datasets/HIT-UAV/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c")
HIT_YOLO = Path("/content/datasets/HIT-UAV-yolo")
img_src = HIT_ROOT / "normal_xml" / "JPEGImages"
ann_src = HIT_ROOT / "normal_xml" / "Annotations"
sets_dir = HIT_ROOT / "normal_xml" / "ImageSets" / "Main"
NAME2YOLO = {"Person": 0, "person": 0, "Car": 3, "car": 3, "Bicycle": 2, "bicycle": 2}

def convert_split(split):
    names = [ln.strip() for ln in (sets_dir / f"{split}.txt").read_text().splitlines() if ln.strip()]
    out_img, out_lab = HIT_YOLO / "images" / split, HIT_YOLO / "labels" / split
    out_img.mkdir(parents=True, exist_ok=True)
    out_lab.mkdir(parents=True, exist_ok=True)
    n = 0
    for stem in names:
        ip, xp = img_src / f"{stem}.jpg", ann_src / f"{stem}.xml"
        if not ip.exists() or not xp.exists():
            continue
        dst = out_img / ip.name
        if not dst.exists():
            try: os.symlink(ip.resolve(), dst)
            except: shutil.copy(ip, dst)
        root = ET.parse(xp).getroot()
        W, H = int(root.find("size/width").text), int(root.find("size/height").text)
        lines = []
        for obj in root.findall("object"):
            cls = NAME2YOLO.get(obj.find("name").text.strip())
            if cls is None: continue
            bb = obj.find("bndbox")
            xmin, ymin = float(bb.find("xmin").text), float(bb.find("ymin").text)
            xmax, ymax = float(bb.find("xmax").text), float(bb.find("ymax").text)
            lines.append(f"{cls} {(xmin+xmax)/2/W:.6f} {(ymin+ymax)/2/H:.6f} {(xmax-xmin)/W:.6f} {(ymax-ymin)/H:.6f}\n")
        (out_lab / f"{stem}.txt").write_text("".join(lines))
        n += 1
    print(split, n)

convert_split("train")
convert_split("val")
convert_split("test")

train 2029
val 290
test 579


In [17]:
from ultralytics import YOLO
from pathlib import Path

HIT_YOLO = Path("/content/datasets/HIT-UAV-yolo")
OUT = Path("/content/dz07_out")
OUT.mkdir(exist_ok=True)

tmp = YOLO("yolov8n.pt")
names_block = "\n".join([f"  {i}: {n}" for i, n in tmp.names.items()])
HIT_YAML = OUT / "hit_uav.yaml"
HIT_YAML.write_text(f"""
path: {HIT_YOLO.resolve()}
train: images/train
val: images/val

names:
{names_block}
""".strip() + "\n")
print(HIT_YAML)
print("val images:", len(list((HIT_YOLO / "images" / "val").glob("*.jpg"))))

# Zero-shot
WEIGHTS = "yolov8n.pt"
for p in [Path("/content/best.pt"), Path("/content/dz07_out/weights/best_dz1.pt")]:
    if p.exists():
        WEIGHTS = str(p)
        break
print("weights:", WEIGHTS)

/content/dz07_out/hit_uav.yaml
val images: 290
weights: yolov8n.pt


In [18]:
from ultralytics import YOLO
from pathlib import Path

HIT_YAML = Path("/content/dz07_out/hit_uav.yaml")
model0 = YOLO("yolov8n.pt")
res0 = model0.val(data=str(HIT_YAML), imgsz=640, conf=0.25, iou=0.7, split="val")
print("ZERO mAP50:", float(res0.box.map50))
print("ZERO mAP50-95:", float(res0.box.map))
print("ZERO P:", float(res0.box.mp))
print("ZERO R:", float(res0.box.mr))

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1734.9±428.6 MB/s, size: 61.1 KB)
val: Scanning /content/datasets/HIT-UAV-yolo/labels/val... 290 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 290/290 1.4Kit/s 0.2s
val: New cache created: /content/datasets/HIT-UAV-yolo/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 4.5it/s 4.2s
                   all        290       2441      0.128    0.00713    0.00476    0.00195
                person        171       1168      0.385     0.0214     0.0143    0.00586
                   car         53        554          0          0          0          0
            motorcycle        136        719          0          0          0          0
Speed: 1.8ms preprocess, 4.7ms inference, 0.0ms loss, 1.5m

## 4. Інтервенція A — grayscale VisDrone subset

Гіпотеза: зсув частково через відсутність кольору.

In [19]:
# Потрібен VisDrone train.
from pathlib import Path
import cv2, random, shutil
from ultralytics import YOLO

VD = Path("/content/datasets/VisDrone/VisDrone2019-DET-val")
img_dir = VD / "images"
# ann: annotations often next to images
ann_dir = VD / "annotations"
if not ann_dir.exists():
    ann_dir = VD / "labels"
print("ann_dir", ann_dir, "exists", ann_dir.exists())
print("sample", list(ann_dir.glob("*"))[:5] if ann_dir.exists() else None)

# VisDrone txt: l,t,w,h,score,cat,trunc,occ  cat:1 ped,2 people,3 bike,4 car
VD2YOLO = {1: 0, 2: 0, 3: 2, 4: 3}  # person, person, bicycle, car

lab_yolo = VD / "labels_yolo"
lab_yolo.mkdir(exist_ok=True)
nlab = 0
if ann_dir.exists():
    for ap in ann_dir.glob("*.txt"):
        im = img_dir / (ap.stem + ".jpg")
        if not im.exists():
            continue
        im0 = cv2.imread(str(im))
        if im0 is None:
            continue
        H, W = im0.shape[:2]
        lines = []
        for line in ap.read_text().splitlines():
            p = line.strip().split(",")
            if len(p) < 6:
                continue
            l, t, w, h = map(float, p[:4])
            cat = int(p[5])
            cls = VD2YOLO.get(cat)
            if cls is None:
                continue
            xc, yc = (l + w/2)/W, (t + h/2)/H
            bw, bh = w/W, h/H
            if bw <= 0 or bh <= 0:
                continue
            lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
        (lab_yolo / (ap.stem + ".txt")).write_text("".join(lines))
        nlab += 1
print("converted labels:", nlab)

# rebuild gray with labels
GRAY = Path("/content/datasets/VisDrone-gray-subset")
shutil.rmtree(GRAY, ignore_errors=True)
(GRAY/"images"/"train").mkdir(parents=True)
(GRAY/"labels"/"train").mkdir(parents=True)
imgs = sorted(img_dir.glob("*.jpg"))
pick = random.Random(42).sample(imgs, min(80, len(imgs)))
for ip in pick:
    g = cv2.cvtColor(cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE), cv2.COLOR_GRAY2BGR)
    cv2.imwrite(str(GRAY/"images"/"train"/ip.name), g)
    lp = lab_yolo/(ip.stem+".txt")
    out = GRAY/"labels"/"train"/(ip.stem+".txt")
    if lp.exists():
        shutil.copy(lp, out)
    else:
        out.write_text("")
print("gray:", len(list((GRAY/"images"/"train").glob("*.jpg"))),
      "nonempty labels:",
      sum(1 for p in (GRAY/"labels"/"train").glob("*.txt") if p.stat().st_size > 0))

OUT = Path("/content/dz07_out")
tmp = YOLO("yolov8n.pt")
names_block = "\n".join([f"  {i}: {n}" for i, n in tmp.names.items()])
(OUT/"visdrone_gray.yaml").write_text(f"""
path: {GRAY.resolve()}
train: images/train
val: images/train
names:
{names_block}
""".strip()+"\n")

model_a = YOLO("yolov8n.pt")
model_a.train(data=str(OUT/"visdrone_gray.yaml"), imgsz=416, epochs=5, batch=4,
              workers=2, seed=42, device="cpu",
              project=str(OUT/"runs"), name="A_gray", exist_ok=True)
best = OUT/"runs"/"A_gray"/"weights"/"best.pt"
if best.exists():
    shutil.copy(best, OUT/"weights"/"A_best.pt")
    print("A_best OK (with labels)")

ann_dir /content/datasets/VisDrone/VisDrone2019-DET-val/annotations exists True
sample [PosixPath('/content/datasets/VisDrone/VisDrone2019-DET-val/annotations/0000330_00801_d_0000804.txt'), PosixPath('/content/datasets/VisDrone/VisDrone2019-DET-val/annotations/0000327_03401_d_0000728.txt'), PosixPath('/content/datasets/VisDrone/VisDrone2019-DET-val/annotations/0000330_01001_d_0000805.txt'), PosixPath('/content/datasets/VisDrone/VisDrone2019-DET-val/annotations/0000271_00801_d_0000378.txt'), PosixPath('/content/datasets/VisDrone/VisDrone2019-DET-val/annotations/0000287_03001_d_0000774.txt')]
converted labels: 548
gray: 80 nonempty labels: 80
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, co

## 5. Інтервенція B — HIT train subset (з ваг ДЗ1)

Гіпотеза: мала in-domain вибірка закриває зсув.

In [20]:

from pathlib import Path
import random, shutil, os
from ultralytics import YOLO

SEED, N, IMGSZ, EPOCHS = 42, 100, 416, 5
OUT = Path("/content/dz07_out")
HIT_YOLO = Path("/content/datasets/HIT-UAV-yolo")
src_img = HIT_YOLO / "images" / "train"
src_lab = HIT_YOLO / "labels" / "train"

sub = Path("/content/datasets/HIT-subset-B")
shutil.rmtree(sub, ignore_errors=True)
(sub/"images"/"train").mkdir(parents=True)
(sub/"labels"/"train").mkdir(parents=True)

imgs = sorted(src_img.glob("*.jpg"))
pick = random.Random(SEED).sample(imgs, min(N, len(imgs)))
for ip in pick:
    dst = sub/"images"/"train"/ip.name
    try: os.symlink(ip.resolve(), dst)
    except: shutil.copy(ip, dst)
    lp = src_lab/(ip.stem+".txt")
    out = sub/"labels"/"train"/(ip.stem+".txt")
    shutil.copy(lp, out) if lp.exists() else out.write_text("")
print("B subset:", len(pick))

names = "\n".join([f"  {i}: {n}" for i,n in YOLO("yolov8n.pt").names.items()])
HIT_SUB_YAML = OUT/"hit_subset_B.yaml"
HIT_SUB_YAML.write_text(f"path: {sub.resolve()}\ntrain: images/train\nval: images/train\nnames:\n{names}\n")

model_b = YOLO("yolov8n.pt")  # з ДЗ1/zero, НЕ з A
model_b.train(data=str(HIT_SUB_YAML), imgsz=IMGSZ, epochs=EPOCHS, batch=4,
              device="cpu", workers=2, seed=SEED,
              project=str(OUT/"runs"), name="B_hit", exist_ok=True)
best = OUT/"runs"/"B_hit"/"weights"/"best.pt"
shutil.copy(best, OUT/"weights"/"B_best.pt")
print("B_best OK")

B subset: 100
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dz07_out/hit_subset_B.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=B_hit, nbs=64, nms=None, opset=None,

## 6. Інтервенція A+B — HIT subset з ваг після A

In [21]:
from pathlib import Path
import shutil
from ultralytics import YOLO

OUT = Path("/content/dz07_out")
model_ab = YOLO(str(OUT / "weights" / "A_best.pt"))
model_ab.train(
    data=str(OUT / "hit_subset_B.yaml"),
    imgsz=416, epochs=5, batch=4,
    device="cpu", workers=2, seed=42,
    project=str(OUT / "runs"), name="AB_hit", exist_ok=True
)
shutil.copy(OUT / "runs" / "AB_hit" / "weights" / "best.pt",
            OUT / "weights" / "AB_best.pt")
print("AB_best OK")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dz07_out/hit_subset_B.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/dz07_out/weights/A_best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=AB_hit, nbs=64, nms=None,

## 7. Повторне оцінювання zero / A / B / A+B на HIT val

In [22]:

from pathlib import Path
from ultralytics import YOLO
import pandas as pd

OUT = Path("/content/dz07_out")
HIT_YAML = OUT / "hit_uav.yaml"

runs = {
    "zero": "yolov8n.pt",
    "A": str(OUT / "weights" / "A_best.pt"),
    "B": str(OUT / "weights" / "B_best.pt"),
    "A+B": str(OUT / "weights" / "AB_best.pt"),
}

rows = []
for name, w in runs.items():
    if name != "zero" and not Path(w).exists():
        print("missing", name, w)
        continue
    m = YOLO(w)
    r = m.val(data=str(HIT_YAML), imgsz=416, conf=0.25, iou=0.7, split="val", verbose=False)
    rows.append({
        "run": name,
        "mAP50": round(float(r.box.map50), 4),
        "mAP50-95": round(float(r.box.map), 4),
        "P": round(float(r.box.mp), 4),
        "R": round(float(r.box.mr), 4),
    })
    print(name, rows[-1])

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(OUT / "metrics_all.csv", index=False)
print("Saved", OUT / "metrics_all.csv")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1942.7±571.5 MB/s, size: 71.2 KB)
val: Scanning /content/datasets/HIT-UAV-yolo/labels/val.cache... 290 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 290/290 76.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 8.3it/s 2.3s
                   all        290       2441     0.0635    0.00114   0.000667   0.000279
Speed: 0.7ms preprocess, 2.4ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/detect/val-2
zero {'run': 'zero', 'mAP50': 0.0007, 'mAP50-95': 0.0003, 'P': 0.0635, 'R': 0.0011}
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image 

### Висновок (шаблон 6–10 речень)
1. Zero-shot майже нуль → сильний domain shift RGB→IR.

2. A (grayscale) майже не допомагає → зсув не лише через відсутність кольору.

3. B (мала in-domain HIT вибірка) різко піднімає mAP50 (~0.10).

4. A+B найкращий (mAP50 0.23) → grayscale-pretrain дає додатковий виграш над чистим B.

5. Для теми 6 логічно лишити ваги A+B.

6. Обмеження: CPU/короткі епохи (5), subset 80–100 кадрів, без GPU на train.